# InferenceOpt — results analysis

Load JSON artifacts from `../results/` (produced by `benchmarks/*.py` and `quantization/bench_quant_compare.py`).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
RES = ROOT / "results"
list(RES.glob("*.json")) if RES.exists() else print("No results dir yet — run benchmarks first.")

In [ ]:
def load_json(name: str):
    p = RES / name
    if not p.exists():
        print(f"missing: {p}")
        return None
    return json.loads(p.read_text(encoding="utf-8"))

lat = load_json("bench_latency.json")
thr = load_json("bench_throughput.json")
pfx = load_json("bench_prefix_cache.json")
qcmp = load_json("bench_quant_compare.json")

In [ ]:
if thr and "levels" in thr:
    df = pd.DataFrame(thr["levels"])
    display(df[["concurrency", "tokens_per_s", "requests_per_s", "wall_s"]])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(df["concurrency"], df["tokens_per_s"], marker="o")
    ax.set_xlabel("Concurrency")
    ax.set_ylabel("Tokens / s")
    ax.set_title("Throughput sweep")
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
if qcmp and "fp16" in qcmp and "awq" in qcmp:
    rows = []
    for label, branch in (("fp16", qcmp["fp16"]), ("awq", qcmp["awq"])):
        latp = branch.get("latency", {}).get("ttft_ms_percentiles", {})
        thr_levels = branch.get("throughput", {}).get("levels", [])
        peak_tps = max((x.get("tokens_per_s") or 0) for x in thr_levels) if thr_levels else float("nan")
        rows.append({"run": label, **latp, "peak_tokens_per_s": peak_tps})
    display(pd.DataFrame(rows))